# Sky Image & Movie

Demonstrates `model.sky_image()` for resolved imaging and animation of GRB afterglows.

**Prerequisites:**
```
pip install matplotlib
```

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

from VegasAfterglow import TophatJet, ISM, Observer, Radiation, Model
from VegasAfterglow.units import uas

In [ ]:
model = Model(
    jet=TophatJet(theta_c=0.1, E_iso=1e52, Gamma0=200),
    medium=ISM(n_ism=1),
    observer=Observer(lumi_dist=1e26, z=0.1, theta_obs=0),
    fwd_rad=Radiation(eps_e=1e-1, eps_B=1e-3, p=2.3),
)

## Single Frame

In [ ]:
img = model.sky_image([1e6], nu_obs=1e9, fov=500 * uas, npixel=512)

print(img)
print(f"pixel scale: {img.extent[1] / 64 / uas:.1f} uas/pix")

fig, ax = plt.subplots(dpi=100)
extent = img.extent / uas  # convert to microarcseconds

im = ax.imshow(
    img.image[0].T,
    origin="lower",
    extent=extent,
    cmap="inferno",
    norm=LogNorm(),
)
ax.set_xlabel(r"$\Delta x$ ($\mu$as)")
ax.set_ylabel(r"$\Delta y$ ($\mu$as)")
ax.set_title(r"$t_{\rm obs} = 10^6$ s, $\nu = 1$ GHz")
fig.colorbar(im, label=r"Surface brightness (erg/cm$^2$/s/Hz/sr)")
plt.tight_layout()

## Movie

Batch call: dynamics solved once, each frame only re-renders the sky projection.

In [ ]:
times = np.logspace(4, 8, 30)  # 30 frames from 10^4 to 10^8 s
imgs = model.sky_image(times, nu_obs=1e9, fov=2000 * uas, npixel=512)
print(f"image shape: {imgs.image.shape}")

extent = imgs.extent / uas

# Find global brightness range for consistent colorbar
vmin = imgs.image[imgs.image > 0].min()
vmax = imgs.image.max()

fig, ax = plt.subplots(dpi=100)
im = ax.imshow(
    imgs.image[0].T,
    origin="lower",
    extent=extent,
    cmap="inferno",
    norm=LogNorm(vmin=vmin, vmax=vmax),
)
title = ax.set_title("")
ax.set_xlabel(r"$\Delta x$ ($\mu$as)")
ax.set_ylabel(r"$\Delta y$ ($\mu$as)")
fig.colorbar(im, label=r"erg/cm$^2$/s/Hz/sr")


def update(frame):
    im.set_data(imgs.image[frame].T)
    title.set_text(f"$t_{{\\rm obs}}$ = {times[frame]:.1e} s")
    return (im, title)


anim = FuncAnimation(fig, update, frames=len(times), interval=100, blit=True)
anim.save("sky-image.gif", writer="pillow", fps=10)
plt.close(fig)
HTML(anim.to_jshtml())

## Off-Axis Jet

For an off-axis observer, the image centroid drifts (superluminal apparent motion).

In [ ]:
model_offaxis = Model(
    jet=TophatJet(theta_c=0.1, E_iso=1e52, Gamma0=200),
    medium=ISM(n_ism=1),
    observer=Observer(lumi_dist=1e26, z=0.1, theta_obs=0.4),
    fwd_rad=Radiation(eps_e=1e-1, eps_B=1e-3, p=2.3),
)

times_oa = np.logspace(5, 8, 30)
imgs_oa = model_offaxis.sky_image(times_oa, nu_obs=1e9, fov=3000 * uas, npixel=512)

extent_oa = imgs_oa.extent / uas
vmin_oa = imgs_oa.image[imgs_oa.image > 0].min()
vmax_oa = imgs_oa.image.max()

fig, ax = plt.subplots(dpi=100)
im = ax.imshow(
    imgs_oa.image[0].T,
    origin="lower",
    extent=extent_oa,
    cmap="inferno",
    norm=LogNorm(vmin=vmin_oa, vmax=vmax_oa),
)
title = ax.set_title("")
ax.set_xlabel(r"$\Delta x$ ($\mu$as)")
ax.set_ylabel(r"$\Delta y$ ($\mu$as)")
fig.colorbar(im, label=r"erg/cm$^2$/s/Hz/sr")


def update_oa(frame):
    im.set_data(imgs_oa.image[frame].T)
    title.set_text(f"off-axis $t_{{\\rm obs}}$ = {times_oa[frame]:.1e} s")
    return (im, title)


anim_oa = FuncAnimation(fig, update_oa, frames=len(times_oa), interval=100, blit=True)
plt.close(fig)
HTML(anim_oa.to_jshtml())

## Flux from Image vs Direct Calculation

Integrating the surface brightness over all pixels (sum × pixel solid angle) recovers the total flux density, which should match `flux_density_grid()`.

In [ ]:
t_obs = np.logspace(3, 8, 30)
nu_obs = 1e9

# Method 1: integrate sky image
img = model.sky_image(t_obs, nu_obs=nu_obs, fov=2000 * uas, npixel=128)
flux_from_image = img.image.sum(axis=(1, 2)) * img.pixel_solid_angle

# Method 2: direct flux density calculation
flux_direct = model.flux_density_grid(t_obs, np.array([nu_obs])).total[0, :]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(5, 5), dpi=150, sharex=True,
                                gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05})

ax1.loglog(t_obs, flux_direct, "k-", label="flux_density_grid")
ax1.loglog(t_obs, flux_from_image, "o", ms=4, color="C1", label="sky_image (integrated)")
ax1.set_ylabel(r"Flux density (erg/cm$^2$/s/Hz)")
ax1.legend()

ratio = flux_from_image / flux_direct
ax2.semilogx(t_obs, ratio, "o-", ms=4, color="C1")
ax2.axhline(1, color="k", ls="--", lw=0.8)
ax2.set_ylabel("image / direct")
ax2.set_xlabel("Observer time (s)")
ax2.set_ylim(0.99, 1.01)
plt.tight_layout()